In [17]:
# Carga de datos.
import pandas as pd
df = pd.read_csv('train.csv')
from sklearn.preprocessing import OneHotEncoder

# LLENAR DATOS VACIOS 
df.loc[df['ESTU_VALORMATRICULAUNIVERSIDAD'].isnull(), 'ESTU_VALORMATRICULAUNIVERSIDAD'] = 'No pago matricula'
df.loc[df['ESTU_HORASSEMANATRABAJA'].isnull(), 'ESTU_HORASSEMANATRABAJA'] = '0'
df.loc[df['ESTU_PAGOMATRICULAPROPIO'].isnull(), 'ESTU_PAGOMATRICULAPROPIO'] = 'No'
df.loc[df['FAMI_ESTRATOVIVIENDA'].isnull(), 'FAMI_ESTRATOVIVIENDA'] = 'Sin Estrato'


# Corregir caracteres especiales
df['ESTU_PRGM_ACADEMICO'] = df['ESTU_PRGM_ACADEMICO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df['ESTU_PRGM_DEPARTAMENTO'] = df['ESTU_PRGM_DEPARTAMENTO'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df['ESTU_VALORMATRICULAUNIVERSIDAD'] = df['ESTU_VALORMATRICULAUNIVERSIDAD'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
df['ESTU_HORASSEMANATRABAJA'] = df['ESTU_HORASSEMANATRABAJA'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')


# Comparar informacion en ambos campos de familia tiene internet
mask_both_yes = (df['FAMI_TIENEINTERNET'] == 'Si') & (df['FAMI_TIENEINTERNET.1'] == 'Si')
mask_one_no = ((df['FAMI_TIENEINTERNET'] == 'No') & (df['FAMI_TIENEINTERNET.1'] == 'Si')) | ((df['FAMI_TIENEINTERNET'] == 'Si') & (df['FAMI_TIENEINTERNET.1'] == 'No'))
df.drop(columns=['FAMI_TIENEINTERNET.1'], inplace=True)

# Eliminar columnas que no sirven
df.drop(columns=['FAMI_TIENELAVADORA','FAMI_TIENEAUTOMOVIL','ESTU_PRIVADO_LIBERTAD','FAMI_EDUCACIONMADRE','FAMI_EDUCACIONPADRE'], inplace=True)

#S
#One Hot Encoding para ESTU_VALORMATRICULAUNIVERSIDAD
categorias = ['No pago matricula', 'Menos de 500 mil', 'Entre 500 mil y menos de 1 millon',
              'Entre 1 millon y menos de 2.5 millones', 'Entre 2.5 millones y menos de 4 millones',
              'Entre 4 millones y menos de 5.5 millones', 'Entre 5.5 millones y menos de 7 millones',
              'Mas de 7 millones']
# Crear un codificador OneHotEncoder para 
encoder = OneHotEncoder(categories=[categorias])
# Transformar las categorías en una matriz one-hot
one_hot_encoded = encoder.fit_transform(df[['ESTU_VALORMATRICULAUNIVERSIDAD']])
# Crear un DataFrame con las columnas one-hot
one_hot_df = pd.DataFrame(one_hot_encoded.toarray(), columns=categorias)
# Concatenar el DataFrame original con el DataFrame one-hot
df = pd.concat([df, one_hot_df], axis=1)
# Eliminar la columna original 'ESTU_VALORMATRICULAUNIVERSIDAD'
df.drop(columns=['ESTU_VALORMATRICULAUNIVERSIDAD'], inplace=True)



#One Hot Encoding para RENDIMIENTO_GLOBAL
categorias_rg = ['alto', 'bajo', 'medio-bajo',
              'medio-alto']
# Crear un codificador OneHotEncoder 
encoder_rg = OneHotEncoder(categories=[categorias_rg])
# Transformar las categorías en una matriz one-hot
one_hot_encoded_rg = encoder_rg.fit_transform(df[['RENDIMIENTO_GLOBAL']])
# Crear un DataFrame con las columnas one-hot
one_hot_df_rg = pd.DataFrame(one_hot_encoded_rg.toarray(), columns=categorias_rg)
# Concatenar el DataFrame original con el DataFrame one-hot
df = pd.concat([df, one_hot_df_rg], axis=1)
# Eliminar la columna original 'RENDIMIENTO_GLOBAL'
df.drop(columns=['RENDIMIENTO_GLOBAL'], inplace=True)


ESTU_HORASSEMANATRABAJA_dummy = pd.get_dummies(df['ESTU_HORASSEMANATRABAJA'], prefix='HORASTRABAJA').astype(int)
df = pd.concat([df, ESTU_HORASSEMANATRABAJA_dummy], axis=1)
df.drop(columns=['ESTU_HORASSEMANATRABAJA'], inplace=True)

#F
mapeo = {
    'Sin Estrato': 0,
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6
}


# Aplicar el mapeo a la columna 'FAMI_ESTRATOVIVIENDA'
df['FAMI_ESTRATOVIVIENDA'] = df['FAMI_ESTRATOVIVIENDA'].map(mapeo)

mapeo = {
    'No': 0,
    'Si': 1,
}

df['FAMI_TIENEINTERNET'] = df['FAMI_TIENEINTERNET'].map(mapeo)
df['ESTU_PAGOMATRICULAPROPIO'] = df['ESTU_PAGOMATRICULAPROPIO'].map(mapeo)
df['FAMI_TIENECOMPUTADOR'] = df['FAMI_TIENECOMPUTADOR'].map(mapeo)


df

,ID,PERIODO,ESTU_PRGM_ACADEMICO,ESTU_PRGM_DEPARTAMENTO,FAMI_ESTRATOVIVIENDA,FAMI_TIENEINTERNET,ESTU_PAGOMATRICULAPROPIO,FAMI_TIENECOMPUTADOR,No pago matricula,Menos de 500 mil,...,Mas de 7 millones,alto,bajo,medio-bajo,medio-alto,HORASTRABAJA_0,HORASTRABAJA_Entre 11 y 20 horas,HORASTRABAJA_Entre 21 y 30 horas,HORASTRABAJA_Mas de 30 horas,HORASTRABAJA_Menos de 10 horas
0,904256,20212,ENFERMERIA,BOGOTA,3,1.0,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0,0,0,0,1
1,645256,20212,DERECHO,ATLANTICO,3,0.0,0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1,0,0,0,0
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTA,3,1.0,0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0,0,0,1,0
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,4,1.0,0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1,0,0,0,0
4,989032,20212,PSICOLOGIA,ANTIOQUIA,3,1.0,0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692495,25096,20195,BIOLOGIA,LA GUAJIRA,2,1.0,1,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0,1,0,0,0
692496,754213,20212,PSICOLOGIA,NORTE SANTANDER,3,1.0,0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0,0,0,1,0
692497,504185,20183,ADMINISTRACION EN SALUD OCUPACIONAL,BOGOTA,3,1.0,1,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0,0,0,0,1
692498,986620,20195,PSICOLOGIA,TOLIMA,1,0.0,1,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0,0,0,0,1
